# Actividad 06 – Ítem C: Faster R-CNN en **video** (simulación de tiempo real)
**Visión Artificial – UNA Puno – X ciclo, grupo B**

Procesa un video **cuadro por cuadro** con Faster R-CNN, guarda el video anotado y registra por frame: latencia, FPS, nº de detecciones y confianza. Mismo formato de salida que el ítem D (YOLO).

**Parámetros comunes del equipo (deben ser idénticos en A, B, C y D):** `CONF=0.25`, `IMGSZ=640`, mismos videos e imágenes (COCO128). **Usen exactamente el mismo video que el ítem D.**

> Usa GPU: *Entorno de ejecución → Cambiar tipo de entorno → T4 GPU*.

## 1. Instalación

In [ ]:
# torch y torchvision ya vienen en Colab
!pip -q install pandas matplotlib

## 2. Modelo Faster R-CNN

In [ ]:
import time, cv2, numpy as np

import torch, torchvision
from torchvision.models.detection import (fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights,
                                          fasterrcnn_mobilenet_v3_large_fpn, FasterRCNN_MobileNet_V3_Large_FPN_Weights)

MODELO_RCNN = "resnet50_v2"      # o "mobilenet_v3" (mucho más rápido, menos preciso)
CONF  = 0.25
IMGSZ = 640
device = 0 if torch.cuda.is_available() else "cpu"
dev = torch.device("cuda" if device == 0 else "cpu")
print("Dispositivo:", torch.cuda.get_device_name(0) if device == 0 else "CPU")

# box_score_thresh bajo (0.001) para poder calcular mAP; CONF se aplica después.
kw = dict(box_score_thresh=0.001, box_detections_per_img=100, min_size=IMGSZ, max_size=int(IMGSZ*1.67))
if MODELO_RCNN == "resnet50_v2":
    w = FasterRCNN_ResNet50_FPN_V2_Weights.COCO_V1
    model = fasterrcnn_resnet50_fpn_v2(weights=w, **kw)
else:
    w = FasterRCNN_MobileNet_V3_Large_FPN_Weights.COCO_V1
    model = fasterrcnn_mobilenet_v3_large_fpn(weights=w, **kw)
model.eval().to(dev)

# Mapeo de ids COCO-91 (torchvision) -> índices COCO-80 (YOLO/COCO128)
cats = w.meta["categories"]
ids80 = [i for i, c in enumerate(cats) if c not in ("__background__", "N/A")]
id2idx = {i: k for k, i in enumerate(ids80)}
nombres = [cats[i] for i in ids80]

@torch.no_grad()
def detectar_rcnn(img_bgr):
    """Devuelve boxes xyxy, scores, labels (0-79) y latencia extremo a extremo en segundos."""
    t0 = time.perf_counter()
    x = torch.from_numpy(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)).permute(2, 0, 1).float().div(255).to(dev)
    o = model([x])[0]
    if device == 0: torch.cuda.synchronize()
    dt = time.perf_counter() - t0
    b = o["boxes"].cpu().numpy(); s = o["scores"].cpu().numpy()
    l = np.array([id2idx[int(i)] for i in o["labels"].cpu().numpy()], dtype=np.int64)
    return b, s, l, dt

MAX_FRAMES = 300   # None = todo el video (usar el mismo valor que en D)
VIDEO_PATH = "video_entrada.mp4"

## 3. Cargar el video
Sube **el mismo video** que usó el equipo de YOLO.

In [ ]:
import os
from google.colab import files
USAR_VIDEO_PROPIO = True   # False = descargar video de ejemplo

if USAR_VIDEO_PROPIO:
    up = files.upload()
    os.replace(list(up.keys())[0], VIDEO_PATH)
else:
    !wget -q -O video_entrada.mp4 https://github.com/intel-iot-devkit/sample-videos/raw/master/person-bicycle-car-detection.mp4

cap = cv2.VideoCapture(VIDEO_PATH)
print("Resolución:", int(cap.get(3)), "x", int(cap.get(4)), "| FPS original:", cap.get(cv2.CAP_PROP_FPS),
      "| Frames:", int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
cap.release()

## 4. Detección frame a frame

In [ ]:
import pandas as pd
cap = cv2.VideoCapture(VIDEO_PATH)
fps_in = cap.get(cv2.CAP_PROP_FPS) or 30
W, H = int(cap.get(3)), int(cap.get(4))
out = cv2.VideoWriter("salida_raw.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps_in, (W, H))

detectar_rcnn(np.zeros((H, W, 3), np.uint8))   # warm-up

rows, class_counts, idx = [], {}, 0
t_global = time.perf_counter()
while True:
    ok, frame = cap.read()
    if not ok or (MAX_FRAMES and idx >= MAX_FRAMES): break
    b, s, l, dt = detectar_rcnn(frame)
    m = s >= CONF
    ann = frame.copy()
    for (x1, y1, x2, y2), sc, c in zip(b[m], s[m], l[m]):
        class_counts[nombres[c]] = class_counts.get(nombres[c], 0) + 1
        cv2.rectangle(ann, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
        cv2.putText(ann, f"{nombres[c]} {sc:.2f}", (int(x1), max(15, int(y1)-5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    cv2.putText(ann, f"{1/dt:.1f} FPS | {int(m.sum())} obj", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    out.write(ann)
    rows.append({"frame": idx, "latencia_ms": dt*1000, "fps": 1/dt, "n_detecciones": int(m.sum()),
                 "conf_media": float(s[m].mean()) if m.any() else np.nan})
    idx += 1
cap.release(); out.release()
df = pd.DataFrame(rows)
print(f"Procesados {idx} frames en {time.perf_counter()-t_global:.1f}s")

## 5. Ver el video resultante

In [ ]:
!ffmpeg -y -loglevel error -i salida_raw.mp4 -vcodec libx264 -pix_fmt yuv420p salida_rcnn.mp4
from IPython.display import HTML
from base64 import b64encode
data = b64encode(open("salida_rcnn.mp4", "rb").read()).decode()
HTML(f'<video width=640 controls><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>')

## 6. Métricas (insumo para el ítem E)

In [ ]:
import matplotlib.pyplot as plt, json
resumen = {"modelo": f"faster-rcnn-{MODELO_RCNN}", "imgsz": IMGSZ, "conf": CONF, "dispositivo": str(device),
    "frames": len(df), "FPS_promedio": round(df.fps.mean(), 2), "FPS_mediana": round(df.fps.median(), 2),
    "latencia_media_ms": round(df.latencia_ms.mean(), 2), "latencia_p95_ms": round(df.latencia_ms.quantile(0.95), 2),
    "detecciones_por_frame": round(df.n_detecciones.mean(), 2), "confianza_media": round(df.conf_media.mean(), 3)}
print(pd.Series(resumen).to_string())

fig, ax = plt.subplots(1, 3, figsize=(16, 3.5))
ax[0].plot(df.fps); ax[0].set_title("FPS por frame")
ax[1].plot(df.latencia_ms); ax[1].set_title("Latencia (ms)")
ax[2].plot(df.n_detecciones); ax[2].set_title("Detecciones por frame")
for a in ax: a.set_xlabel("frame")
plt.tight_layout(); plt.savefig("metricas_video_rcnn.png", dpi=150); plt.show()

pd.Series(class_counts).sort_values(ascending=False).head(10).plot.barh(title="Clases detectadas (top 10)")
plt.gca().invert_yaxis(); plt.show()

df.to_csv("metricas_video_rcnn.csv", index=False)
json.dump(resumen, open("resumen_video_rcnn.json", "w"), indent=1)

## 7. Descargar resultados

In [ ]:
for f in ["salida_rcnn.mp4", "metricas_video_rcnn.csv", "metricas_video_rcnn.png", "resumen_video_rcnn.json"]:
    files.download(f)